# Klygo Data Mining & YOLOv8 Curation Benchmark

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/integration/Mining_Curation_YOLO_Benchmark.ipynb)

This notebook walks through **all 13 public APIs** of `klygo.mining` (8 operational functions + 5 structured report containers) and runs an end-to-end **YOLOv8 (`yolov8n.pt`) training & validation benchmark** comparing **100% full training data** against **every data selection and curation strategy** on a fixed validation split.

> **Colab Recommendation:** Select **Runtime → Change runtime type → T4 GPU** to accelerate YOLOv8 training across all candidate subsets.

## Jump to an API
- **Submodule 1 — `klygo.mining.analysis`**:
  - [`mining.stats` & `mining.MiningStats`](#2-dataset-profiling-miningstats--miningminingstats): dataset-wide counts, box area percentiles, class co-occurrence, imbalance ratio, and 6 image quality metrics.
  - [`mining.score` & `mining.MiningScores`](#3-multi-criteria-image-scoring-miningscore--miningminingscores): composite `[0.0, 1.0]` per-image value ranking (`quality`, `rarity`, `coverage`, `boxes`).
  - [`mining.compare` & `mining.MiningComparison`](#8-before- vs-after-dataset-comparison-miningcompare--miningminingcomparison): side-by-side statistical delta verification before and after curation.
- **Submodule 2 — `klygo.mining.duplicates`**:
  - [`mining.find_duplicates`, `mining.DuplicateReport`, & `mining.deduplicate`](#4-exact--perceptual-deduplication-find_duplicates-duplicatereport-deduplicate): MD5 exact + 64-bit DCT perceptual hash (`phash`) duplicate group detection and whole-image deduplication.
- **Submodule 3 — `klygo.mining.curation`**:
  - [`mining.cluster` & `mining.ClusterResult`](#5-multimodal-k-means-clustering-miningcluster--miningclusterresult): HSV histogram + HOG edge + quality + box geometry + class presence K-Means++ clustering.
  - [`mining.select` (All 9 Strategies)](#6-all-9-whole-image-selection-strategies-miningselect): `random`, `weighted`, `quality`, `rare_first`, `diversity`, `boxes`, `quota`, `coverage`, and `class`.
  - [`mining.balance`](#7-class-distribution-balancing-miningbalance): whole-image class frequency balancing (`min`, `median`, `mean`) without synthetic duplication.
- **End-to-End YOLOv8 Benchmark**:
  - [`YOLOv8n Training & Leaderboard`](#9-yolov8n-training-benchmark-curated-subsets-vs-original-dataset): train `yolov8n.pt` on each curated subset and compare `Precision`, `Recall`, `mAP@50`, `mAP@50-95`, and training speed against the full dataset and random baseline.

## 1. Environment Setup & Standard Dataset Preparation

Install `klygo` (`v2-release` branch) and `ultralytics` for YOLOv8 training and validation.

In [ ]:
!pip install --upgrade --no-cache-dir git+https://github.com/IchigoMazone/klygo.git@v2-release ultralytics

### Download and Partition the Dataset with `klygo.files`, `klygo.archive`, and `klygo.datasets`

We download **COCO128** (128 diverse multi-class images — or switch `DATASET_NAME = 'coco8'` for an instant smoke test), normalize it via `datasets.convert()`, and split it with `datasets.partition()` into:
- **`train_pool` (75%)**: The candidate pool where `klygo.mining` curation algorithms operate.
- **`val_fixed` (25%)**: A strictly held-out validation set shared across all YOLOv8 training runs for a fair **mAP@50** and **mAP@50-95** comparison.

In [ ]:
import time
import torch
from rich.console import Console
from rich.table import Table

from klygo import archive, config, datasets, files, media, mining, outputs, visual

console = Console(width=125)
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
console.print(f'[bold green]Active Compute Device:[/bold green] {DEVICE}')

# Set DATASET_NAME = 'coco128' for full benchmark or 'coco8' for a quick smoke test
DATASET_NAME = 'coco128'
workspace = files.join('artifacts', 'colab_mining_benchmark')
files.remove(workspace, missing_ok=True)
files.mkdir(workspace)

zip_url = f'https://github.com/ultralytics/assets/releases/download/v0.0.0/{DATASET_NAME}.zip'
yaml_url = f'https://raw.githubusercontent.com/ultralytics/ultralytics/main/ultralytics/cfg/datasets/{DATASET_NAME}.yaml'

zip_file = files.download(zip_url, workspace, overwrite=True, verbose=True)
yaml_file = files.download(yaml_url, workspace, overwrite=True, verbose=True)

raw_dir = files.join(workspace, 'raw')
archive.extract(zip_file, raw_dir, overwrite=True, verbose=False)
files.copy(yaml_file, files.join(raw_dir, f'{DATASET_NAME}.yaml'), overwrite=True)

# Convert raw archive into a normalized Flat YOLO Dataset
image_sources = [f'{DATASET_NAME}/images/train2017'] if DATASET_NAME == 'coco128' else [f'{DATASET_NAME}/images/train', f'{DATASET_NAME}/images/val']
label_sources = [f'{DATASET_NAME}/labels/train2017'] if DATASET_NAME == 'coco128' else [f'{DATASET_NAME}/labels/train', f'{DATASET_NAME}/labels/val']

all_flat = datasets.convert(
    source=raw_dir,
    target=files.join(workspace, 'all_flat'),
    tree={
        'images': image_sources,
        'labels': label_sources,
        'metadata': f'{DATASET_NAME}.yaml',
    },
    on_conflict='rename',
    overwrite=True,
)

# Partition 75% into train_pool (for mining) and 25% into val_fixed (for evaluation)
split_dir = datasets.partition(
    all_flat,
    files.join(workspace, 'split_base'),
    ratios=(0.75, 0.25, 0.0),
    seed=42,
    verbose=False,
)

train_flat = datasets.convert(
    source=split_dir,
    target=files.join(workspace, 'train_pool'),
    tree={'images': 'images/train', 'labels': 'labels/train', 'metadata': 'data.yaml'},
    overwrite=True,
)
val_flat = datasets.convert(
    source=split_dir,
    target=files.join(workspace, 'val_fixed'),
    tree={'images': 'images/val', 'labels': 'labels/val', 'metadata': 'data.yaml'},
    overwrite=True,
)

# Load train_pool and val_fixed into standardized klygo.outputs.Detections containers
data = datasets.load(train_flat, format='yolo', verbose=True)
val_data = datasets.load(val_flat, format='yolo', verbose=False)
console.print(f'[bold cyan]Train Pool:[/bold cyan] {len(data)} images ({data.total_objects} boxes) | [bold yellow]Fixed Val:[/bold yellow] {len(val_data)} images ({val_data.total_objects} boxes)')

---
## 2. Dataset Profiling: `mining.stats()` & `mining.MiningStats`

### API Overview
- **`mining.stats(data, *, quality=True, cooccurrence=True) -> MiningStats`**:
  - Computes total `images`, `boxes`, `classes`, `empty_images` (background frames without annotations), and `multilabel_images` (frames containing 2+ distinct classes).
  - Tallies `images_per_class`, `boxes_per_class`, and normalized relative `box_area` percentiles (`min`, `median`, `mean`, `max`).
  - Builds the symmetric class `cooccurrence` matrix and computes `imbalance_ratio = max_class_count / min_class_count`.
  - When `quality=True`, evaluates 6 classical visual quality metrics across all images: `brightness`, `contrast`, `sharpness`, `entropy`, `noise`, and `exposure`.
- **`mining.MiningStats`**:
  - Structured report returned by `mining.stats()`, supporting `.show(console)` for Rich terminal tables, `.to_dict()`, and `.save(path)` (`.json`, `.md`, `.csv`).

In [ ]:
stats_report = mining.stats(data, quality=True, cooccurrence=True)
stats_report.show(console)

console.print(f'Imbalance Ratio: [bold red]{stats_report.imbalance_ratio:.2f}[/bold red]')
console.print('Box Area Percentiles:', stats_report.box_area)

# Export MiningStats report in all 3 supported formats (.json, .md, .csv)
reports_dir = files.join(workspace, 'reports')
stats_report.save(files.join(reports_dir, '01_stats.json'))
stats_report.save(files.join(reports_dir, '01_stats.md'))
stats_report.save(files.join(reports_dir, '01_stats.csv'))

---
## 3. Multi-Criteria Image Scoring: `mining.score()` & `mining.MiningScores`

### API Overview
- **`mining.score(data, *, weights=None) -> MiningScores`**:
  - Assigns every image a normalized composite score in `[0.0, 1.0]` combining four weighted criteria:
    1. `quality` (default `0.4`): sharpness, contrast, entropy, low noise, and balanced exposure.
    2. `rarity` (default `0.3`): inverse class frequency reward for images containing underrepresented classes.
    3. `coverage` (default `0.2`): ratio of distinct classes present in the image.
    4. `boxes` (default `0.1`): normalized annotation density relative to the dataset.
- **`mining.MiningScores`**:
  - Stores `.values` (composite scores), `.details` (per-criterion breakdown dicts), `.weights`, and `.identities`.

In [ ]:
scores_report = mining.score(
    data,
    weights={'quality': 0.4, 'rarity': 0.3, 'coverage': 0.2, 'boxes': 0.1},
)
scores_report.show(console)

best_idx = scores_report.values.index(max(scores_report.values))
worst_idx = scores_report.values.index(min(scores_report.values))
console.print(f'[bold green]Highest-Scored Image:[/bold green] index={best_idx}, score={scores_report.values[best_idx]:.4f}, breakdown={scores_report.details[best_idx]}')
console.print(f'[bold yellow]Lowest-Scored Image:[/bold yellow] index={worst_idx}, score={scores_report.values[worst_idx]:.4f}, breakdown={scores_report.details[worst_idx]}')

scores_report.save(files.join(reports_dir, '02_scores.csv'))

---
## 4. Exact & Perceptual Deduplication: `find_duplicates()`, `DuplicateReport`, `deduplicate()`

### API Overview
- **`mining.find_duplicates(data, *, exact=True, method='phash', threshold=0.95) -> DuplicateReport`**:
  - Two-stage non-destructive duplicate scanner: (1) MD5 byte-level file digest grouping (`exact_groups`), and (2) 64-bit DCT perceptual hash (`phash`) grouping for near-duplicates with similarity $\ge$ `threshold` (`groups`).
- **`mining.DuplicateReport`**:
  - Exposes `.groups`, `.exact_groups`, `.similarities`, `.hashes`, and `.duplicate_images` (count of redundant frames).
- **`mining.deduplicate(data, *, keep='best', threshold=0.95) -> Detections`**:
  - Returns a curated `Detections` container keeping one representative image per duplicate cluster according to `keep`:
    - `'best'`: highest composite `mining.score()` and annotation count.
    - `'quality'`: highest visual clarity score.
    - `'annotations'`: highest bounding box count.
    - `'first'`: earliest index in dataset order.

In [ ]:
dup_report = mining.find_duplicates(data, exact=True, method='phash', threshold=0.92)
dup_report.show(console)
console.print(f'Duplicate Groups: {len(dup_report.groups)} | Redundant Images: {dup_report.duplicate_images}')
dup_report.save(files.join(reports_dir, '03_duplicates.json'))

# Test all 4 retention policies of mining.deduplicate
dedup_best = mining.deduplicate(data, keep='best', threshold=0.92)
dedup_quality = mining.deduplicate(data, keep='quality', threshold=0.92)
dedup_annot = mining.deduplicate(data, keep='annotations', threshold=0.92)
dedup_first = mining.deduplicate(data, keep='first', threshold=0.92)

console.print(f'After deduplicate(keep="best"): {len(dedup_best)}/{len(data)} images retained.')

---
## 5. Multimodal K-Means Clustering: `mining.cluster()` & `mining.ClusterResult`

### API Overview
- **`mining.cluster(data, *, features=('histogram', 'hog', 'quality', 'boxes', 'classes'), clusters=100, method='kmeans', seed=42) -> ClusterResult`**:
  - Extracts multimodal feature vectors combining HSV color histograms (`histogram`), HOG edge orientation histograms (`hog`), 6 visual quality descriptors (`quality`), box geometry statistics (`boxes`), and class presence (`classes`).
  - Applies Z-score standardization and deterministic K-Means++ clustering (`seed`).
  - Identifies the centroid-nearest representative image (`representatives`) for each cluster to power diversity sampling.
- **`mining.ClusterResult`**:
  - Contains `.labels`, `.groups`, `.representatives`, and `.feature_vectors`.

In [ ]:
NUM_CLUSTERS = min(12, len(dedup_best))
cluster_report = mining.cluster(
    dedup_best,
    clusters=NUM_CLUSTERS,
    features=('histogram', 'hog', 'quality', 'boxes', 'classes'),
    method='kmeans',
    seed=42,
)
cluster_report.show(console)
console.print('Cluster Centroid Representatives:', cluster_report.representatives)
cluster_report.save(files.join(reports_dir, '04_clusters.json'))

---
## 6. All 9 Whole-Image Selection Strategies: `mining.select()`

### API Overview
- **`mining.select(data, *, method='weighted', limit=None, ...) -> Detections`**:
  Supports **9 atomic whole-image selection strategies** (never stripping individual boxes from selected frames):
  1. **`'random'`**: Reproducible uniform random sampling controlled by `seed`.
  2. **`'weighted'`**: Top-ranked images by composite `MiningScores`.
  3. **`'quality'`**: Images exceeding a minimum visual quality score `threshold`.
  4. **`'rare_first'`**: Prioritizes images containing the rarest object classes first.
  5. **`'diversity'`**: Round-robin sampling across `ClusterResult` clusters starting from centroids.
  6. **`'boxes'`**: Filters frames matching `[min_boxes, max_boxes]` and `min_area_ratio`.
  7. **`'quota'`**: Greedy set-cover selection fulfilling at least `per_class` images per label.
  8. **`'coverage'`**: Maximizes unique class coverage before filling remaining budget.
  9. **`'class'`**: Filters frames by class names using `match` modes `'any'`, `'all'`, `'only'`, or `'exclude'`.

In [ ]:
# Set a 50% image subset budget for fair comparison across selection methods
BUDGET = max(4, len(dedup_best) // 2)
dedup_scores = mining.score(dedup_best)

selection_showcase = {
    '1. random':        mining.select(dedup_best, method='random', limit=BUDGET, seed=42),
    '2. weighted':      mining.select(dedup_best, method='weighted', scores=dedup_scores, limit=BUDGET),
    '3. quality':       mining.select(dedup_best, method='quality', scores=dedup_scores, threshold=0.25, limit=BUDGET),
    '4. rare_first':    mining.select(dedup_best, method='rare_first', scores=dedup_scores, limit=BUDGET),
    '5. diversity':     mining.select(dedup_best, method='diversity', cluster_result=cluster_report, per_cluster=max(1, BUDGET // NUM_CLUSTERS + 1), limit=BUDGET),
    '6. boxes':         mining.select(dedup_best, method='boxes', min_boxes=2, max_boxes=30, min_area_ratio=0.001, limit=BUDGET),
    '7. quota':         mining.select(dedup_best, method='quota', per_class=2, limit=BUDGET),
    '8. coverage':      mining.select(dedup_best, method='coverage', per_class=1, limit=BUDGET),
    '9a. class (any)':  mining.select(dedup_best, method='class', classes=['person'], match='any', limit=BUDGET),
    '9b. class (excl)': mining.select(dedup_best, method='class', classes=['person'], match='exclude', limit=BUDGET),
}

sel_table = Table(title=f'Summary of All 9 mining.select Strategies (Budget = {BUDGET} Images)')
sel_table.add_column('Strategy', style='cyan')
sel_table.add_column('Images', justify='right', style='green')
sel_table.add_column('Total Boxes', justify='right', style='yellow')
sel_table.add_column('Classes Retained', justify='right', style='magenta')

for label, subset in selection_showcase.items():
    st = mining.stats(subset, quality=False, cooccurrence=False)
    sel_table.add_row(label, str(st.images), str(st.boxes), str(st.classes))

console.print(sel_table)

---
## 7. Class Distribution Balancing: `mining.balance()`

### API Overview
- **`mining.balance(data, *, target='median', strategy='select', count_by='images', seed=42) -> Detections`**:
  - Curates a whole-image subset reducing class frequency skew toward `target` (`'min'`, `'median'`, `'mean'`, or an explicit integer cap) without duplicating images.

In [ ]:
balanced_min = mining.balance(dedup_best, target='min', strategy='select', count_by='images', seed=42)
balanced_median = mining.balance(dedup_best, target='median', strategy='select', count_by='images', seed=42)
balanced_mean = mining.balance(dedup_best, target='mean', strategy='select', count_by='images', seed=42)

console.print(
    f'Balanced Subsets -> target=min: {len(balanced_min)} images | '
    f'target=median: {len(balanced_median)} images | '
    f'target=mean: {len(balanced_mean)} images'
)

---
## 8. Before-vs-After Dataset Comparison: `mining.compare()` & `mining.MiningComparison`

### API Overview
- **`mining.compare(before, after, *, quality=False) -> MiningComparison`**:
  - Computes side-by-side `MiningStats` reports (`before` and `after`) and exact delta metrics in `.changes` (`images`, `boxes`, `classes`, `imbalance_ratio`).

In [ ]:
comp_report = mining.compare(data, selection_showcase['2. weighted'])
comp_report.show(console)
console.print('Comparison Deltas:', comp_report.changes)
comp_report.save(files.join(reports_dir, '05_comparison_weighted.md'))

---
## 9. YOLOv8n Training Benchmark: Curated Subsets vs Original Dataset

Now we stage each subset into a standard YOLO directory structure sharing the full class registry, train a fresh **YOLOv8n (`yolov8n.pt`)** model under identical hyperparameters (`EPOCHS`, `IMGSZ`, `BATCH`, `SEED=42`), and evaluate every trained model on the held-out **`val_fixed`** dataset.

### Benchmark Contenders:
1. **`01_full_original`**: 100% of `train_pool` (Upper-bound full-data baseline).
2. **`02_deduplicated`**: Dataset after perceptual deduplication (`mining.deduplicate`).
3. **`03_select_random`**: Uniform random subset of `BUDGET` images (`method='random'`).
4. **`04_select_weighted`**: Multi-criteria scored subset of `BUDGET` images (`method='weighted'`).
5. **`05_select_diversity`**: K-Means cluster-diverse subset of `BUDGET` images (`method='diversity'`).
6. **`06_select_rare_first`**: Rare-class-prioritized subset of `BUDGET` images (`method='rare_first'`).
7. **`07_select_quality`**: Visual-quality-filtered subset of `BUDGET` images (`method='quality'`).
8. **`08_select_boxes`**: High-density bounding-box subset of `BUDGET` images (`method='boxes'`).
9. **`09_select_quota`**: Class-quota set-cover subset of `BUDGET` images (`method='quota'`).
10. **`10_select_coverage`**: Class-coverage-maximized subset of `BUDGET` images (`method='coverage'`).
11. **`11_balanced_median`**: Class-balanced subset (`mining.balance(target='median')`).

In [ ]:
from ultralytics import YOLO

def stage_yolo_training_dataset(
    train_subset: outputs.Detections,
    source_flat_dir: str,
    val_dataset_dir: str,
    output_dir: str,
) -> str:
    """Export a Detections subset into a YOLO training directory aligned with the full class registry."""
    files.remove(output_dir, missing_ok=True)
    images_dir = files.join(output_dir, 'images')
    labels_dir = files.join(output_dir, 'labels')
    files.mkdir(images_dir)
    files.mkdir(labels_dir)

    base_meta = config.load(files.join(source_flat_dir, 'data.yaml'), verbose=False).to_dict()
    names = {int(k): str(v) for k, v in base_meta['names'].items()}
    label_to_id = {label: cid for cid, label in names.items()}

    for idx, frame in enumerate(train_subset):
        stem = f'{idx:05d}_{files.stem(frame.source_path)}'
        ext = files.extension(frame.source_path).lower()
        files.copy(frame.source_path, files.join(images_dir, stem + ext), overwrite=True)

        w, h = frame.image_size
        lines = [
            f'{label_to_id[box.label]} {((box.xmin + box.xmax) / 2) / w:.6f} {((box.ymin + box.ymax) / 2) / h:.6f} {box.width / w:.6f} {box.height / h:.6f}'
            for box in frame
        ]
        files.save(files.join(labels_dir, stem + '.txt'), lines, overwrite=True, verbose=False)

    yaml_path = files.join(output_dir, 'data.yaml')
    meta = {
        'path': files.as_posix(files.resolve(output_dir)),
        'train': 'images',
        'val': files.as_posix(files.join(files.resolve(val_dataset_dir), 'images')),
        'nc': len(names),
        'names': names,
    }
    config.save(yaml_path, meta, overwrite=True, verbose=False)
    return str(files.resolve(yaml_path))

# Shared YOLOv8 training hyperparameters across all methods
EPOCHS = 15
IMGSZ = 640
BATCH = 16
SEED = 42
runs_project_dir = str(files.resolve(files.join(workspace, 'yolo_runs')))

benchmark_candidates = {
    '01_full_original':    data,
    '02_deduplicated':     dedup_best,
    '03_select_random':    selection_showcase['1. random'],
    '04_select_weighted':  selection_showcase['2. weighted'],
    '05_select_diversity': selection_showcase['5. diversity'],
    '06_select_rare_first':selection_showcase['4. rare_first'],
    '07_select_quality':   selection_showcase['3. quality'],
    '08_select_boxes':     selection_showcase['6. boxes'],
    '09_select_quota':     selection_showcase['7. quota'],
    '10_select_coverage':  selection_showcase['8. coverage'],
    '11_balanced_median':  balanced_median,
}

leaderboard = []

for method_name, subset in benchmark_candidates.items():
    if len(subset) == 0:
        continue
    console.rule(f'[bold cyan]Training YOLOv8n on: {method_name} ({len(subset)} images)[/bold cyan]')
    staged_dir = files.join(workspace, 'staged_datasets', method_name)
    data_yaml = stage_yolo_training_dataset(subset, train_flat, val_flat, staged_dir)
    sub_stats = mining.stats(subset, quality=False, cooccurrence=False)

    start_t = time.perf_counter()
    model = YOLO('yolov8n.pt')
    model.train(
        data=data_yaml,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=min(BATCH, max(1, len(subset))),
        seed=SEED,
        device=DEVICE,
        project=runs_project_dir,
        name=method_name,
        exist_ok=True,
        verbose=False,
    )
    val_metrics = model.val(
        data=data_yaml,
        split='val',
        device=DEVICE,
        project=runs_project_dir,
        name=f'{method_name}_val',
        exist_ok=True,
        verbose=False,
    )
    elapsed_s = time.perf_counter() - start_t

    leaderboard.append({
        'method': method_name,
        'images': sub_stats.images,
        'image_ratio': f'{(sub_stats.images / len(data)) * 100:.1f}%',
        'boxes': sub_stats.boxes,
        'classes_kept': sub_stats.classes,
        'imbalance': round(sub_stats.imbalance_ratio, 2),
        'precision': round(float(val_metrics.box.mp), 4),
        'recall': round(float(val_metrics.box.mr), 4),
        'mAP50': round(float(val_metrics.box.map50), 4),
        'mAP50_95': round(float(val_metrics.box.map), 4),
        'train_time_s': round(elapsed_s, 2),
    })

### Benchmark Leaderboard Table & Report Export

Rank all curation strategies by `mAP@50-95` on `val_fixed` and compute the exact gain/loss relative to both the **Random Baseline (`03_select_random`)** and the **Full Dataset (`01_full_original`)**.

In [ ]:
# Extract baseline mAP@50-95 scores for delta calculation
random_map = next((r['mAP50_95'] for r in leaderboard if r['method'] == '03_select_random'), 0.0)
full_map = next((r['mAP50_95'] for r in leaderboard if r['method'] == '01_full_original'), 0.0)

# Sort leaderboard by mAP50_95 descending
ranked_leaderboard = sorted(leaderboard, key=lambda r: (r['mAP50_95'], r['mAP50']), reverse=True)

bench_table = Table(title=f'Klygo Data Mining x YOLOv8n Benchmark (Val Fixed = {len(val_data)} images, Epochs = {EPOCHS})')
bench_table.add_column('Rank', justify='right', style='bold white')
bench_table.add_column('Method', style='bold cyan')
bench_table.add_column('Train Imgs (%)', justify='right', style='green')
bench_table.add_column('Boxes', justify='right', style='yellow')
bench_table.add_column('Classes', justify='right', style='magenta')
bench_table.add_column('Imbalance', justify='right')
bench_table.add_column('Precision', justify='right')
bench_table.add_column('Recall', justify='right')
bench_table.add_column('mAP@50', justify='right', style='bold green')
bench_table.add_column('mAP@50-95', justify='right', style='bold yellow')
bench_table.add_column('vs Random', justify='right')
bench_table.add_column('Time (s)', justify='right')

for rank, row in enumerate(ranked_leaderboard, 1):
    delta_rand = row['mAP50_95'] - random_map
    delta_str = f'{delta_rand:+.4f}'
    row['delta_vs_random'] = round(delta_rand, 4)
    row['delta_vs_full'] = round(row['mAP50_95'] - full_map, 4)
    bench_table.add_row(
        str(rank),
        row['method'],
        f"{row['images']} ({row['image_ratio']})",
        str(row['boxes']),
        str(row['classes_kept']),
        f"{row['imbalance']:.2f}",
        f"{row['precision']:.4f}",
        f"{row['recall']:.4f}",
        f"{row['mAP50']:.4f}",
        f"{row['mAP50_95']:.4f}",
        delta_str,
        f"{row['train_time_s']:.1f}s",
    )

console.print(bench_table)

# Save leaderboard to JSON and CSV reports
files.save(files.join(reports_dir, 'yolo_mining_leaderboard.json'), ranked_leaderboard, overwrite=True)
files.save(files.join(reports_dir, 'yolo_mining_leaderboard.csv'), ranked_leaderboard, overwrite=True)
console.print(f'[bold green]Saved benchmark leaderboard reports to:[/bold green] {reports_dir}')

## Remember

- `klygo.mining` operates strictly at the **whole-image level**, ensuring selected training subsets never corrupt per-image object annotations.
- Always evaluate curated training subsets against an untouched validation split (`val_fixed`) sharing the original class registry (`names`).
- Explore the submodule tutorials: [Mining Overview](../mining/Document_Mining_Overview.ipynb) ([Colab](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/mining/Document_Mining_Overview.ipynb)), [Analysis](../mining/Document_Mining_Analysis.ipynb), [Duplicates](../mining/Document_Mining_Duplicates.ipynb), and [Curation](../mining/Document_Mining_Curation.ipynb).